# Historical P4 small-model reference

These preserved executed outputs are the earlier 24-run experiment. The final team notebook is now P1-owned at `llm-test/ai_prototype.ipynb`; P4 larger-model evidence is separate in `person4_large_results_review.ipynb`. Do not aggregate both P4 result sets without a deliberate model-selection plan.

# Hestia A6 — Person 4 model tests and UI integration

This contribution tests **three additional public models** not present in P1/P2's
completed CSVs: Qwen2-0.5B-Instruct, SmolLM-360M-Instruct, and Qwen3-0.6B.
They are different from teammates' Qwen2.5 and SmolLM2 IDs. Selection follows the
user's authorization to choose any three uncompleted models, bounded by this
machine's approximately 15.5 GiB RAM and lack of a CUDA GPU.

These small models **do not complete the missing large-model categories** in the
team's original roster. Person 1 owns the final 15-model matrix; keep the root
notebook intact and merge this CSV through its existing merge cell.

Set up a separate Python 3.11 environment using
`docs/a6-person-4/requirements-benchmark.txt`. Weights stay in ignored
`model_cache/`. No API keys or paid calls are needed.

## Download weights first

The first code cell downloads the actual public Transformers safetensors and
tokenizers into an ordinary local directory (avoids Windows symlink privileges).
It records publisher revision IDs and reuses existing local files on subsequent
runs. First execution requires network access and a few GB of disk space.

In [1]:
from pathlib import Path
import importlib.util
import sys

ROOT = Path.cwd()
if not (ROOT / 'manage.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'manage.py').exists(), 'Run from repository root or llm-test/'
spec = importlib.util.spec_from_file_location('p4_benchmark', ROOT / 'docs/a6-person-4/benchmark.py')
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
snapshots = runner.download_models()
{model: details['revision'] for model, details in snapshots.items()}

E:\Hestia\a6-brain-dump\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 373.49it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 337.72it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 372.65it/s]

{'Qwen/Qwen2-0.5B-Instruct': 'c540970f9e29518b1d8f06ab8b24cba66ad77b6d',
 'HuggingFaceTB/SmolLM-360M-Instruct': '73b7144f76331266f5f45d5642fd8da653583b13',
 'Qwen/Qwen3-0.6B': 'c1899de289a04d12100db370d81485cdf75e47ca'}

## Shared experiment contract

Two fixed household paragraphs × four prompting modes × three models = **24
generations**. The runner imports the exact system instruction, examples, parser,
and schema functions from the checked-in P2 helper, aligned to P1's notebook.
`zero/one/few/many` use 0/1/2/3 examples. Generation is greedy, capped at 220 new
tokens, CPU FP32 with four PyTorch threads and seed 0. Qwen3 thinking is explicitly
disabled to keep direct JSON output within the same token budget; this mode and
greedy decoding are disclosed deviations from vendor-recommended sampling.

Model-card sources:
- https://huggingface.co/Qwen/Qwen2-0.5B-Instruct
- https://huggingface.co/HuggingFaceTB/SmolLM-360M-Instruct
- https://huggingface.co/Qwen/Qwen3-0.6B

The runner checkpoints real outputs after each response. If the CSV already
contains all 24 runs, this cell resumes without re-generating; the following
cells display the stored evidence. To rerun from scratch, move the P4 CSV aside
first. Download time is separate from recorded model-load and generation time.
CPU timings are not comparable to teammates' hardware without controls.

In [2]:
team = runner.contract()
print(team.SYSTEM_INSTRUCTION)
print(team.TEST_INPUTS)
print('Shot modes:', team.SHOT_MODES)
runner.run(snapshots)

You are Hestia's household task extraction assistant.
Convert the user's message into separate household tasks.
Return JSON only in this exact shape:
{"tasks":[{"title":"...","task_type":"CHORE|APPOINTMENT|DEADLINE|ERRAND|MAINTENANCE|SHOPPING|OTHER","priority":"LOW|MEDIUM|HIGH|URGENT","description":"..."}]}
Do not invent concerns that the user did not mention. Use URGENT only for an immediate safety risk or a deadline within 24 hours. Keep titles and descriptions concise.
{'BD1': 'Guests are coming tomorrow. The kitchen is messy, we need groceries, and the bathroom tap is leaking badly.', 'BD2': 'This week I need to book the dentist, replace the smoke detector batteries, return a library book by Friday, and wash the guest-room sheets.'}
Shot modes: ['zero', 'one', 'few', 'many']
Saved 24 actual generations to E:\Hestia\a6-brain-dump\a6_results\person4_benchmark_results.csv


## Environment and automatic schema results

Raw text is kept even when JSON fails or the token cap truncates a response.
Schema success is measured separately from semantic quality. A model can produce
valid JSON that copies an example or misses the household's actual concerns.

In [3]:
import json
import pandas as pd
print((ROOT / 'a6_results/person4_environment.json').read_text(encoding='utf-8'))
results = pd.read_csv(ROOT / 'a6_results/person4_benchmark_results.csv')
assert len(results) == 24
assert results.groupby('model_id').size().eq(8).all()
assert set(results['model_id']) == set(runner.MODELS)
summary = results.groupby('model_id').agg(
    runs=('input_id', 'size'),
    schema_success_rate=('schema_valid', 'mean'),
    mean_generation_seconds=('generation_seconds', 'mean'),
    mean_quality=('quality_score_0_to_10', 'mean'),
)
display(summary.round(3))
display(results.groupby(['model_id', 'shot_mode']).agg(
    schema_success_rate=('schema_valid', 'mean'),
    mean_generation_seconds=('generation_seconds', 'mean'),
    mean_quality=('quality_score_0_to_10', 'mean'),
).round(3))

{
  "generated_at_utc": "2026-10-19T01:18:52.472580+00:00",
  "python": "3.11.15",
  "torch": "2.14.1+cpu",
  "transformers": "4.57.6",
  "platform": "Windows-10-10.0.26200-SP0",
  "device": "cpu",
  "dtype": "float32",
  "torch_threads": 4,
  "ram_gib": 15.5,
  "logical_processors": 18,
  "max_new_tokens": 220,
  "do_sample": false,
  "seed": 0,
  "models": {
    "Qwen/Qwen2-0.5B-Instruct": {
      "revision": "c540970f9e29518b1d8f06ab8b24cba66ad77b6d"
    },
    "HuggingFaceTB/SmolLM-360M-Instruct": {
      "revision": "73b7144f76331266f5f45d5642fd8da653583b13"
    },
    "Qwen/Qwen3-0.6B": {
      "revision": "c1899de289a04d12100db370d81485cdf75e47ca"
    }
  },
  "qwen3_thinking": false,
  "team_contract_sha256": "0a5cb229532575a932dbe520844861933bfdface6c3e49b3a436ceed7ca39abd",
  "notes": "Shared P1/P2 inputs, instructions, examples and schema. Qwen3 uses non-thinking chat template; greedy decoding follows team contract rather than vendor sampling guidance. CPU timings are not co

,runs,schema_success_rate,mean_generation_seconds,mean_quality
model_id,,,,
HuggingFaceTB/SmolLM-360M-Instruct,8,0.000,26.817,0.000
Qwen/Qwen2-0.5B-Instruct,8,0.125,18.891,5.125
Qwen/Qwen3-0.6B,8,0.875,23.354,6.375


schema_success_rate  \
model_id                           shot_mode                        
HuggingFaceTB/SmolLM-360M-Instruct few                        0.0   
                                   many                       0.0   
                                   one                        0.0   
                                   zero                       0.0   
Qwen/Qwen2-0.5B-Instruct           few                        0.0   
                                   many                       0.5   
                                   one                        0.0   
                                   zero                       0.0   
Qwen/Qwen3-0.6B                    few                        1.0   
                                   many                       1.0   
                                   one                        1.0   
                                   zero                       0.5   

                                              mean_generation_seconds  \
model_id                           shot_mode                            
HuggingFaceTB/SmolLM-360M-Instruct few                         26.155   
                                   many                        27.938   
                                   one                         26.974   
                                   zero                        26.200   
Qwen/Qwen2-0.5B-Instruct           few                         20.916   
                                   many                        21.981   
                                   one                         15.651   
                                   zero                        17.015   
Qwen/Qwen3-0.6B                    few                         27.511   
                                   many                        29.069   
                                   one                         18.783   
                                   zero                        18.054   

                                              mean_quality  
model_id                           shot_mode                
HuggingFaceTB/SmolLM-360M-Instruct few                 0.0  
                                   many                0.0  
                                   one                 0.0  
                                   zero                0.0  
Qwen/Qwen2-0.5B-Instruct           few                 5.0  
                                   many                6.0  
                                   one                 5.0  
                                   zero                4.5  
Qwen/Qwen3-0.6B                    few                 7.0  
                                   many                6.5  
                                   one                 8.0  
                                   zero                4.0

## Review every raw response

Shared rubric: extraction 0–3, task types 0–2, priority 0–2, clarity 0–2,
JSON format 0–1. Scores and written notes are assistant-assisted content reviews,
not a separate blinded human study. The submitter should inspect the raw outputs
and approve/correct them. Invalid JSON does not erase partially correct semantic
content, but such output is rejected by Django rather than saved.


In [4]:
for row in results.to_dict('records'):
    print('\n' + '=' * 75)
    print(row['model_id'], row['input_id'], row['shot_mode'])
    print('Generation seconds:', row['generation_seconds'], 'Schema:', row['schema_valid'])
    print(row['raw_output'])
    print('Score:', row['quality_score_0_to_10'], 'Review:', row['review_notes'])


Qwen/Qwen2-0.5B-Instruct BD1 zero
Generation seconds: 14.99 Schema: False
{"tasks":[{"title":"Kitchen Cleaning","task_type":"CLEANING","priority":"MEDIUM","description":"Clean the kitchen to make it ready for guests.","urgency":"LOW"},{"title":"Groceries Shopping","task_type":"SHOPPING","priority":"MEDIUM","description":"Buy groceries for the guests.","urgency":"MEDIUM"},{"title":"Bathroom Repair","task_type":"MAINTENANCE","priority":"HIGH","description":"Fix the leaky bathroom faucet so the guests can come home safely."},{"title":"Other Tasks","task_type":"OTHER","priority":"LOW","description":"Other household chores like dusting, vacuuming, etc."}]}
Score: 5 Review: Assistant-assisted review; E/T/P/C/F=2/1/1/1/0. Found the three concerns but invented additional chores; CLEANING is invalid, adds urgency keys, and understates guest preparation.

Qwen/Qwen2-0.5B-Instruct BD1 one
Generation seconds: 15.099 Schema: False
{"tasks": [{"title": "Vacuum the kitchen", "task_type": "CLOTHES", 

## Findings, limitations, and handoff

See the generated measured summary below, the full CSV, and revision metadata.
No results from unrun models are invented. The test has only two inputs and
greedy decoding; it cannot establish broad quality, production concurrency, or
the final team-wide size sweet spot. All generated tasks still require UI review.


In [5]:
from IPython.display import Markdown, display
display(Markdown((ROOT / 'docs/a6-person-4/benchmark_summary.md').read_text(encoding='utf-8')))
print('P1 handoff:', 'a6_results/person4_benchmark_results.csv')

# P4 measured benchmark findings

Completed October 18, 2026 on CPU FP32, four PyTorch threads, approximately 15.5 GiB RAM.
Three distinct additional models × two fixed inputs × four shot modes = **24 real generations**.

| Model | Runs | Schema valid | Mean generation | Mean reviewed quality /10 |
| --- | ---: | ---: | ---: | ---: |
| Qwen/Qwen2-0.5B-Instruct | 8 | 1/8 | 18.891 s | 5.125 |
| HuggingFaceTB/SmolLM-360M-Instruct | 8 | 0/8 | 26.817 s | 0.000 |
| Qwen/Qwen3-0.6B | 8 | 7/8 | 23.354 s | 6.375 |

## Prompting comparison

| Model | Mode | Schema valid | Mean quality /10 |
| --- | --- | ---: | ---: |
| Qwen/Qwen2-0.5B-Instruct | zero | 0/2 | 4.50 |
| Qwen/Qwen2-0.5B-Instruct | one | 0/2 | 5.00 |
| Qwen/Qwen2-0.5B-Instruct | few | 0/2 | 5.00 |
| Qwen/Qwen2-0.5B-Instruct | many | 1/2 | 6.00 |
| HuggingFaceTB/SmolLM-360M-Instruct | zero | 0/2 | 0.00 |
| HuggingFaceTB/SmolLM-360M-Instruct | one | 0/2 | 0.00 |
| HuggingFaceTB/SmolLM-360M-Instruct | few | 0/2 | 0.00 |
| HuggingFaceTB/SmolLM-360M-Instruct | many | 0/2 | 0.00 |
| Qwen/Qwen3-0.6B | zero | 1/2 | 4.00 |
| Qwen/Qwen3-0.6B | one | 2/2 | 8.00 |
| Qwen/Qwen3-0.6B | few | 2/2 | 7.00 |
| Qwen/Qwen3-0.6B | many | 2/2 | 6.50 |

## What the outputs show

- **Qwen2-0.5B-Instruct:** often recognizes the stated responsibilities but invents invalid task types (CLEANING, PERFORM, RECYCLING) or unsupported detail. Its one schema-valid run still needs type/priority edits; extra urgency keys pass the shared automatic validator but are dropped by the UI.
- **SmolLM-360M-Instruct:** all eight responses return incomplete Python implementation code rather than an input-specific task JSON payload. Several copy prompt examples. This older SmolLM checkpoint is distinct from P1's SmolLM2 checkpoint; do not combine their results under one name.
- **Qwen3-0.6B:** schema reliability alone is misleading: its BD1 zero-shot response uses category names as titles/descriptions instead of actual responsibilities. Other outputs merge or duplicate concerns and overstate urgency. More examples are not a guarantee of better extraction.

No P4 candidate is validated as a production-quality extractor from this small test. Keep mandatory editing/confirmation and retry behavior. Person 2's configured default is preserved; selecting a final team winner requires the full team's results and a broader evaluation.

## Scoring and reproducibility

Each raw response was read and assigned extraction/type/priority/clarity/format points (3/2/2/2/1) in `review_results.py`. CSV notes contain the point breakdown and response-specific reason. These are explicitly **assistant-assisted reviews**, not a blinded human evaluation. The submitter should verify or revise them before submission. The shared automatic schema validator allows extra fields and does not measure relevance.

Model revisions, runtime versions, precision and options are in `a6_results/person4_environment.json`. Downloads are excluded from model-load timings. Greedy generation is capped at 220 tokens; Qwen3 thinking is disabled. CPU timings from this machine cannot establish speed differences versus teammates' hardware.

## Scaling illustration using measured generation time

At 20,000 requests/day, average arrival rate is 0.2315/s. The following is a serialized-worker lower-bound illustration, **not a measured production capacity claim**. Each benchmark worker uses four CPU threads; multiple workers introduce contention and extra model memory.

| Model | Mean generation | Busy workers at average load, rounded up | At 5× peak, rounded up |
| --- | ---: | ---: | ---: |
| Qwen/Qwen2-0.5B-Instruct | 18.891 s | 5 | 22 |
| HuggingFaceTB/SmolLM-360M-Instruct | 26.817 s | 7 | 32 |
| Qwen/Qwen3-0.6B | 23.354 s | 6 | 28 |

See `local-vs-api.md` for sourced token/rental costs and the distinction between billing break-even and serving capacity.

## Team handoff / category limitation

P1 can merge `a6_results/person4_benchmark_results.csv` using the existing notebook CSV merge cell. The root P1 notebook is unchanged. P4's executed notebook is `llm-test/ai_prototype.ipynb`.

These three models were chosen under the user's instruction to use any models not completed by teammates, constrained by this machine. They do not fill the original missing 7–14B/very-large categories. The original roster has category counts 4/3/3/3/2; the assignment requires 3/3/3/3/3. P1/the team must reconcile the final model list and run the missing categories. P4's additional tests do not imply the complete group 15-model requirement is satisfied.


P1 handoff: a6_results/person4_benchmark_results.csv


## Part 3 — local model vs paid API

The detailed estimate is linked below and also referenced in `docs/notes/notes.txt`.
Pricing is sourced and dated; token counts are workload assumptions, not paid API
measurements. The external provider remains Person 3's responsibility.

In [6]:
display(Markdown((ROOT / 'docs/a6-person-4/local-vs-api.md').read_text(encoding='utf-8')))

# Part 3 — P4: Local model vs paid API

Pricing checked October 18, 2026. These are transparent planning estimates, not
paid API experiments or proof that different-sized models have equal quality.
P4 local measured findings are in `benchmark_summary.md` and the benchmark CSV.

## Workload and cost assumptions

Assume 10,000 daily users, two extractions per user per day, 800 input tokens per
extraction (including instructions/examples) and 220 output tokens. That is
20,000 requests, 16M input tokens, and 4.4M output tokens per day. Actual prompt
length varies by shot mode; 800 is a planning scenario, not a measured average.
Thirty days gives 600,000 requests, 480M input and 132M output tokens.

Groq's published GPT-OSS 20B prices are $0.075/M input tokens and $0.30/M output
tokens. GPT-OSS 120B is $0.15/M input and $0.60/M output. These are public text
inference price references, not the team's selected Person 3 provider or a claim
of equivalence to the three sub-billion local models.
[Source: Groq supported models](https://console.groq.com/docs/models).

| Option | Calculation | Estimated USD |
| --- | --- | ---: |
| API 20B, per extraction | (800 × 0.075 + 220 × 0.30) / 1,000,000 | $0.000126 |
| API 20B, daily | 16 × 0.075 + 4.4 × 0.30 | $2.52 |
| API 20B, 30 days | 480 × 0.075 + 132 × 0.30 | $75.60 |
| API 120B, 30 days | 480 × 0.15 + 132 × 0.60 | $151.20 |
| Example RTX 3090 always on | $0.50/hour × 24 × 30 | $360.00 |
| Example RTX 4090 always on | $0.89/hour × 24 × 30 | $640.80 |

Runpod's displayed Pod configurations list RTX 3090 (24 GB VRAM, 125 GB host RAM,
16 vCPUs) at $0.50/hour and RTX 4090 (24 GB VRAM, 41 GB host RAM, 6 vCPUs) at
$0.89/hour. They are example offers, not quotes for the assignment's exact
16-core/64-GB/1-TB configuration. Availability, storage, network, replicas,
monitoring, application hosting, and engineering are additional considerations.
[Source: Runpod pricing](https://www.runpod.io/pricing).

Free tiers and student credits can reduce classroom costs, but rate limits and
expiry make them unsuitable as the only 10,000-user capacity plan. Cached input
discounts, batching, tax, retries and reasoning tokens are omitted. Include all
billed tokens in a real budget. We made no paid API calls for this comparison.

## When a local server is acceptable

Local inference can fit a classroom demonstration, an already-owned machine,
steady predictable demand, offline operation, or a documented requirement to
keep data on infrastructure controlled by the team. Existing hardware has a
marginal electricity cost rather than a new cloud rental bill, but still needs
maintenance and capacity planning. Local inference itself has no per-token bill.

For the assignment's hypothetical 16-core CPU, GPU, 64 GB RAM, and 1 TB disk,
specify the GPU model and **VRAM**: “16-core GPU” alone does not describe model
capacity. Approximate unquantized weights consume 4 bytes/parameter in FP32 and
2 bytes in FP16/BF16; a 7B model is about 28/14 GB, a 14B model about 56/28 GB.
Runtime activations, KV cache, Python, concurrent requests and the OS need extra
memory. A 14B FP16 model therefore exceeds a single 24-GB GPU before overhead.
Four-bit weights are about 0.5 byte/parameter before quantization metadata, but
require supported kernels and a separate quality/latency evaluation. Host RAM is
not interchangeable with GPU VRAM; disk capacity does not make inference fast.

Our measured runs use CPU FP32 on approximately 15.5 GiB RAM and cannot establish
how the hypothetical server or 7–14B models perform. Results support only the
tested small models and two household inputs. There is no honest final
15-model size sweet-spot conclusion until all required categories are measured.

## When an API is preferable

At this illustrative workload, $75.60/month in tokens is below a continuously
rented $360/month GPU, before either option's common Django costs. APIs are
attractive for spiky demand, quick rollout, stronger-model quality, and avoiding
model loading/serving operations, provided acceptable privacy terms, rate limits,
latency and provider reliability are established. A published model size alone
does not prove comparable extraction quality; rerun the same prompts on any
candidate paid model before selecting it.

Using the 20B price example, a $12/day local rental equals token spending at about
95,238 extractions/day: 12 / 0.000126. This is a **billing-only** break-even. It
does not show that one GPU can serve that workload. At 20,000/day, average arrival
rate is 0.2315 requests/second. If local measured mean generation time is L
seconds, serialized service needs at least 0.2315 × L continuously busy workers
before prompt processing, peak headroom and latency constraints. At five times
average peak demand, multiply that by five. Use a production inference server,
queue, bounded concurrency and a real load test rather than running many Django
workers that each load an independent model.

## Hybrid potential

Use the local model to extract ordinary household tasks. Validate output and
require user review; reserve external AI for an explicitly requested breakdown
or a user-approved retry when local extraction fails. Never send raw household
input to an external provider silently. A quality gate can detect malformed
schema but cannot reliably detect all semantic mistakes; do not treat it as a
calibrated model-confidence score.

Example: 20% of the 20,000 daily requests use API extraction, and 10% request a
breakdown costing 300 input/600 output tokens. API extraction costs $0.504/day;
2,000 breakdowns cost (0.6 × 0.075 + 1.2 × 0.30) = $0.405/day. Combined external
cost is $0.909/day, or $27.27/30 days, **plus local hosting**. Adding a rented
$360/month GPU totals $387.27, which exceeds the $75.60 all-API extraction-only
scenario; those workloads differ because the hybrid also includes breakdowns.
Including the same breakdown volume in all-API yields $87.75/month. Hybrid is
therefore not automatically cheaper; it can make sense with already-owned local
capacity, privacy constraints, sustained utilization, or expensive API models.

The implemented P4 UI provides the explicit optional breakdown action, validates
returned steps, preserves edits on errors, and requires fresh confirmation after
adding steps. Person 3 must supply/configure the actual provider and verify its
pricing, limits and external behavior. No paid-provider quality/latency result is
claimed here.
